# Knowledge Center 404 Audit

This notebook enumerates all pages listed in the site sitemap whose URL matches one of the configured source-page prefixes, checks every non-fragment link found on those pages, and exports links that either return an HTTP `404` or redirect to `https://data.niaid.nih.gov/404`.


In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urljoin, urldefrag

import pandas as pd
import requests
from bs4 import BeautifulSoup

SOURCE_PAGE_PREFIXES = [
    'https://data.niaid.nih.gov/knowledge-center',
    'https://data.niaid.nih.gov/about',
    'https://data.niaid.nih.gov/diseases',
    'https://data.niaid.nih.gov/features',
]
SITEMAP_URL = 'https://data.niaid.nih.gov/sitemap-0.xml'
OUTPUT_PATH = 'knowledge_center_404_report.xlsx'
NOT_FOUND_URL = 'https://data.niaid.nih.gov/404'
MAX_WORKERS = 12

session = requests.Session()
session.headers.update({'User-Agent': 'Mozilla/5.0 knowledge-center-link-audit'})


In [2]:
def get_source_pages() -> list[str]:
    response = session.get(SITEMAP_URL, timeout=30)
    response.raise_for_status()
    sitemap = BeautifulSoup(response.text, 'xml')
    urls = []
    for loc in sitemap.find_all('loc'):
        url = loc.get_text(strip=True)
        if any(url.startswith(prefix) for prefix in SOURCE_PAGE_PREFIXES):
            urls.append(url)
    return sorted(dict.fromkeys(urls))


def extract_links(page_url: str) -> list[dict]:
    response = session.get(page_url, timeout=30)
    response.raise_for_status()
    html = BeautifulSoup(response.text, 'html.parser')
    records = []
    seen = set()
    for anchor in html.find_all('a', href=True):
        href = anchor['href'].strip()
        if not href or href.startswith(('#', 'mailto:', 'tel:', 'javascript:')):
            continue
        resolved = urldefrag(urljoin(page_url, href)).url
        key = (page_url, href, resolved)
        if key in seen:
            continue
        seen.add(key)
        records.append({'source_page': page_url, 'original_link': href, 'resolved_link': resolved})
    return records


def fetch_status(record: dict) -> dict:
    try:
        response = session.get(record['resolved_link'], timeout=30, allow_redirects=True)
        final_url = urldefrag(response.url).url.rstrip('/')
        redirected_to_404 = final_url == NOT_FOUND_URL.rstrip('/')
        result = record | {
            'status_code': response.status_code,
            'final_url': response.url,
            'redirected_to_404': redirected_to_404,
            'is_broken': response.status_code == 404 or redirected_to_404,
            'error': '',
        }
    except requests.RequestException as exc:
        result = record | {
            'status_code': None,
            'final_url': '',
            'redirected_to_404': False,
            'is_broken': False,
            'error': str(exc),
        }
    return result


In [3]:
pages = get_source_pages()
print(f'Source pages found: {len(pages)}')

all_links = []
for page in pages:
    all_links.extend(extract_links(page))

print(f'Links collected: {len(all_links)}')

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(fetch_status, record) for record in all_links]
    for future in as_completed(futures):
        results.append(future.result())

results_df = pd.DataFrame(results).sort_values(['source_page', 'original_link']).reset_index(drop=True)
broken_df = results_df.loc[
    results_df['is_broken'],
    ['original_link', 'resolved_link', 'source_page', 'status_code', 'final_url', 'redirected_to_404'],
]
broken_df


Source pages found: 76
Links collected: 1749


,original_link,resolved_link,source_page,status_code,final_url,redirected_to_404
6,/search/q=&from=1&filters=%28%40type%3A%28%22D...,https://data.niaid.nih.gov/search/q=&from=1&fi...,https://data.niaid.nih.gov/about,200,https://data.niaid.nih.gov/404,True
108,Immune Tolerance Network (ITN),https://data.niaid.nih.gov/diseases/Immune Tol...,https://data.niaid.nih.gov/diseases/asthma,200,https://data.niaid.nih.gov/404,True
563,http://www.cdc.gov/std/hpv/stdfact-hpv.htm,http://www.cdc.gov/std/hpv/stdfact-hpv.htm,https://data.niaid.nih.gov/diseases/human-papi...,404,https://www.cdc.gov/std/hpv/stdfact-hpv.htm,False
1438,/knowledge-center/keyed-search,https://data.niaid.nih.gov/knowledge-center/ke...,https://data.niaid.nih.gov/knowledge-center/fr...,200,https://data.niaid.nih.gov/404,True


In [4]:
broken_df.to_excel(OUTPUT_PATH, index=False)
print(f'Wrote {len(broken_df)} broken-link rows to {OUTPUT_PATH}')


Wrote 4 broken-link rows to knowledge_center_404_report.xlsx
